In [2]:

# @title 🔧 階段 1：環境設定與資料庫初始化

import os
import sys
import sqlite3
import subprocess
from datetime import datetime
import pytz

print("=" * 60)
print("🚀 量化專案檔案分析系統 - 穩定版")
print("=" * 60)

# ===== 參數設定 =====
PROJECT_ROOT = "/content/drive/MyDrive/sp_lab_v10 - G"  # @param {type:"string"}
WORK_DIR = "/content/project_analysis"

# ===== 掛載 Google Drive =====
if not os.path.exists("/content/drive"):
    print("\n📂 正在掛載 Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive 已掛載")
else:
    print("\n✅ Google Drive 已就緒")

# ===== 檢查專案路徑 =====
if not os.path.exists(PROJECT_ROOT):
    print(f"\n❌ 錯誤：找不到專案路徑")
    print(f"   {PROJECT_ROOT}")
    print("\n請確認路徑是否正確，或 Drive 是否已掛載。")
    sys.exit(1)
else:
    print(f"✅ 專案路徑確認：{PROJECT_ROOT}")

# ===== 台北時區時間戳（ISO 8601）=====
taipei_tz = pytz.timezone('Asia/Taipei')
timestamp = datetime.now(taipei_tz).isoformat(timespec='seconds')
print(f"📅 執行時間：{timestamp}")

# ===== 建立本地工作目錄 =====
os.makedirs(WORK_DIR, exist_ok=True)
db_path = f"{WORK_DIR}/project_analysis.db"
print(f"📁 本地工作區：{WORK_DIR}")

# ===== 初始化 SQLite 資料庫 =====
print("\n🗄️ 初始化資料庫...")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 檔案基本資訊表
cursor.execute("""
CREATE TABLE IF NOT EXISTS files (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filepath TEXT UNIQUE NOT NULL,
    filename TEXT NOT NULL,
    file_type TEXT,
    file_size INTEGER,
    content_text TEXT,
    char_count INTEGER,
    content_hash TEXT,
    is_duplicate BOOLEAN DEFAULT 0,
    auto_category TEXT,
    preprocessed_at TIMESTAMP,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# 模型分析結果表
cursor.execute("""
CREATE TABLE IF NOT EXISTS analysis (
    file_id INTEGER PRIMARY KEY,
    summary TEXT,
    category TEXT,
    category_id INTEGER,
    keep_decision TEXT,
    reason TEXT,
    confidence TEXT,
    tokens_used INTEGER DEFAULT 0,
    analysis_time_seconds REAL DEFAULT 0,
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (file_id) REFERENCES files(id)
)
""")

# 處理進度追蹤表
cursor.execute("""
CREATE TABLE IF NOT EXISTS progress (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    stage TEXT,
    total_files INTEGER,
    completed_files INTEGER,
    current_file TEXT,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
print("✅ 資料庫結構已建立")
print(f"   位置：{db_path}")

# ===== 安裝必要套件 =====
print("\n📦 檢查必要套件...")
try:
    import pandas
    import tqdm
    import requests
    print("✅ 所有套件已就緒")
except ImportError:
    print("⬇️ 安裝缺少的套件...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "tqdm", "requests"], check=True)
    print("✅ 套件安裝完成")

# ===== 儲存全域變數供後續儲存格使用 =====
globals()['PROJECT_ROOT'] = PROJECT_ROOT
globals()['WORK_DIR'] = WORK_DIR
globals()['DB_PATH'] = db_path
globals()['TIMESTAMP'] = timestamp
globals()['DB_CONN'] = conn

print("\n" + "=" * 60)
print("🎯 階段 1 完成！準備進入檔案預處理階段。")
print("=" * 60)

🚀 量化專案檔案分析系統 - 穩定版

📂 正在掛載 Google Drive...
Mounted at /content/drive
✅ Google Drive 已掛載
✅ 專案路徑確認：/content/drive/MyDrive/sp_lab_v10 - G
📅 執行時間：2025-11-28T01:04:35+08:00
📁 本地工作區：/content/project_analysis

🗄️ 初始化資料庫...
✅ 資料庫結構已建立
   位置：/content/project_analysis/project_analysis.db

📦 檢查必要套件...
✅ 所有套件已就緒

🎯 階段 1 完成！準備進入檔案預處理階段。


In [3]:

# @title 📂 階段 2：檔案掃描與預處理

import os
import json
import hashlib
from datetime import datetime
from tqdm.notebook import tqdm

print("=" * 60)
print("📂 開始檔案掃描與預處理")
print("=" * 60)

# ===== 智慧排除規則 =====
EXCLUDE_PATTERNS = [
    "archive/",
    "lo2cin4bt-main/",
    "output/",
    "__pycache__/",
    ".git/",
    ".ipynb_checkpoints/",
    "node_modules/",
]

EXCLUDE_EXTENSIONS = [
    ".pyc", ".pyo", ".pyd",
    ".db", ".sqlite", ".sqlite3",
    ".png", ".jpg", ".jpeg", ".gif", ".bmp",
    ".mp4", ".avi", ".mov",
    ".zip", ".tar", ".gz",
]

# ===== 自動分類規則（基於路徑）=====
AUTO_CATEGORY_RULES = {
    "archive/": "過時檔案",
    "tests/": "測試程式",
    "test/": "測試程式",
    "docs/": "說明文件",
    "scripts/": "分析工具",
    "src/": "交易策略",
    "research/": "研究筆記",
    "config/": "設定檔案",
    "utils/": "分析工具",
}

# ===== 工具函式 =====
def should_exclude(filepath):
    """檢查是否應排除此檔案"""
    for pattern in EXCLUDE_PATTERNS:
        if pattern in filepath:
            return True

    ext = os.path.splitext(filepath)[1].lower()
    if ext in EXCLUDE_EXTENSIONS:
        return True

    return False

def get_auto_category(filepath):
    """基於路徑自動分類"""
    for path_pattern, category in AUTO_CATEGORY_RULES.items():
        if path_pattern in filepath:
            return category
    return None

def convert_ipynb_to_text(filepath):
    """將 .ipynb 轉換為純文字"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            notebook = json.load(f)

        text_parts = []
        for cell in notebook.get('cells', []):
            cell_type = cell.get('cell_type')
            source = ''.join(cell.get('source', []))

            if cell_type == 'code':
                text_parts.append(f"# CODE CELL\n{source}\n")
            elif cell_type == 'markdown':
                text_parts.append(f"# MARKDOWN\n{source}\n")

        return '\n'.join(text_parts)
    except Exception as e:
        return f"[無法解析 Notebook: {e}]"

def read_file_content(filepath):
    """讀取檔案內容並轉換為純文字"""
    file_ext = os.path.splitext(filepath)[1].lower()

    try:
        if file_ext == '.ipynb':
            return convert_ipynb_to_text(filepath)
        else:
            # 一般文字檔
            encodings = ['utf-8', 'utf-8-sig', 'cp950', 'big5']
            for encoding in encodings:
                try:
                    with open(filepath, 'r', encoding=encoding) as f:
                        return f.read()
                except UnicodeDecodeError:
                    continue

            # 如果所有編碼都失敗，使用 ignore 模式
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                return f.read()
    except Exception as e:
        return f"[讀取錯誤: {e}]"

def calculate_hash(content):
    """計算內容的 SHA256 hash"""
    return hashlib.sha256(content.encode('utf-8')).hexdigest()

# ===== 掃描專案檔案 =====
print("\n🔍 正在掃描專案資料夾...")
all_files = []
target_extensions = ['.py', '.ipynb', '.md', '.txt', '.json', '.yaml', '.yml']

for root, dirs, files in os.walk(PROJECT_ROOT):
    # 修改 dirs 列表來跳過排除的資料夾
    dirs[:] = [d for d in dirs if not should_exclude(os.path.join(root, d))]

    for filename in files:
        filepath = os.path.join(root, filename)
        rel_path = os.path.relpath(filepath, PROJECT_ROOT)

        # 檢查是否應排除
        if should_exclude(rel_path):
            continue

        # 檢查副檔名
        ext = os.path.splitext(filename)[1].lower()
        if ext not in target_extensions:
            continue

        all_files.append({
            'filepath': rel_path,
            'filename': filename,
            'file_type': ext,
            'full_path': filepath
        })

print(f"✅ 找到 {len(all_files)} 個目標檔案")

# ===== 處理檔案並入庫 =====
print("\n📝 開始處理檔案...")
conn = DB_CONN
cursor = conn.cursor()

processed_hashes = {}  # 用於偵測重複檔案
duplicate_count = 0

for file_info in tqdm(all_files, desc="處理檔案", unit="檔"):
    filepath = file_info['filepath']
    filename = file_info['filename']
    file_type = file_info['file_type']
    full_path = file_info['full_path']

    # 讀取檔案內容
    content = read_file_content(full_path)
    char_count = len(content)
    file_size = os.path.getsize(full_path)

    # 計算 hash
    content_hash = calculate_hash(content)

    # 檢查是否重複
    is_duplicate = 0
    if content_hash in processed_hashes:
        is_duplicate = 1
        duplicate_count += 1
    else:
        processed_hashes[content_hash] = filepath

    # 自動分類
    auto_category = get_auto_category(filepath)

    # 存入資料庫
    try:
        cursor.execute("""
        INSERT OR IGNORE INTO files
        (filepath, filename, file_type, file_size, content_text, char_count,
         content_hash, is_duplicate, auto_category, preprocessed_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            filepath, filename, file_type, file_size, content, char_count,
            content_hash, is_duplicate, auto_category, datetime.now().isoformat()
        ))
    except Exception as e:
        print(f"\n⚠️ 無法插入檔案 {filename}: {e}")

conn.commit()

# ===== 統計資訊 =====
print("\n📊 預處理統計：")
print(f"   總檔案數：{len(all_files)}")
print(f"   重複檔案：{duplicate_count}")

cursor.execute("SELECT auto_category, COUNT(*) FROM files WHERE auto_category IS NOT NULL GROUP BY auto_category")
auto_categorized = cursor.fetchall()
print(f"\n   自動分類檔案：")
for cat, count in auto_categorized:
    print(f"      - {cat}: {count}")

cursor.execute("SELECT COUNT(*) FROM files WHERE auto_category IS NULL AND is_duplicate = 0")
need_analysis = cursor.fetchone()[0]
print(f"\n   需模型分析：{need_analysis} 個檔案")

print("\n" + "=" * 60)
print("🎯 階段 2 完成！準備進入模型分析階段。")
print("=" * 60)

📂 開始檔案掃描與預處理

🔍 正在掃描專案資料夾...
✅ 找到 177 個目標檔案

📝 開始處理檔案...


處理檔案:   0%|          | 0/177 [00:00<?, ?檔/s]


📊 預處理統計：
   總檔案數：177
   重複檔案：4

   自動分類檔案：
      - 交易策略: 13
      - 分析工具: 49
      - 測試程式: 10
      - 研究筆記: 3
      - 設定檔案: 5
      - 說明文件: 53

   需模型分析：44 個檔案

🎯 階段 2 完成！準備進入模型分析階段。


In [ ]:

# @title 🤖 階段 3：逐欄位模型分析（穩定模式）

import time
import requests
import subprocess
from tqdm.notebook import tqdm

print("=" * 60)
print("🤖 開始模型分析（逐欄位提問，穩定優先）")
print("=" * 60)

# ===== Ollama 環境檢查與設定 =====
MODEL_NAME = "gemma3:4b"

def ensure_ollama():
    """確保 Ollama 已安裝並啟動"""
    print("\n🔧 檢查 Ollama 環境...")

    # 檢查是否已安裝
    try:
        result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
        print(f"✅ Ollama 已安裝：{result.stdout.strip()}")
    except FileNotFoundError:
        print("⬇️ 安裝 Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        print("✅ Ollama 安裝完成")

    # 啟動伺服器（背景執行）
    print("🚀 啟動 Ollama 伺服器...")
    subprocess.run("pkill ollama", shell=True)
    time.sleep(2)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

    # 確認伺服器就緒
    for _ in range(10):
        try:
            requests.get("http://127.0.0.1:11434", timeout=2)
            print("✅ Ollama 伺服器已就緒")
            break
        except:
            time.sleep(2)

    # 下載模型
    print(f"⬇️ 確保模型 {MODEL_NAME} 已下載...")
    subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
    print("✅ 模型準備完成")

ensure_ollama()

# ===== 模型呼叫函式 =====
def ask_model(prompt, max_retries=3):
    """呼叫 Ollama 模型並返回清理後的回答"""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                "http://127.0.0.1:11434/api/generate",
                json={
                    "model": MODEL_NAME,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.1,
                        "num_ctx": 8192,
                    }
                },
                timeout=120
            )
            response.raise_for_status()
            data = response.json()
            answer = data.get("response", "").strip()
            tokens = data.get("eval_count", 0)
            return answer, tokens
        except Exception as e:
            if attempt == max_retries - 1:
                return "0", 0  # 失敗返回 0
            time.sleep(2)
    return "0", 0

# ===== 分類對照表 =====
CATEGORY_MAP = {
    "0": "無法判斷",
    "1": "交易策略",
    "2": "數據處理",
    "3": "分析工具",
    "4": "設定檔案",
    "5": "說明文件",
    "6": "研究筆記",
    "7": "測試程式",
    "8": "過時檔案",
    "9": "第三方程式",
}

# ===== 提示詞設計（帶上下文連貫）=====

def get_prompt_summary(filename, filepath, content):
    """欄位 1：Summary 提示詞"""
    return f"""你是專業的程式碼分析助手。請用一句繁體中文描述這個檔案的主要功能。

要求：
- 不超過 30 個字
- 不要說「這個檔案是...」開頭，直接說功能
- 如果無法判斷，回答：0

檔案名稱：{filename}
檔案路徑：{filepath}
檔案內容（前 2000 字）：
{content[:2000]}

你的一句話描述（不超過30字，無法判斷寫0）："""

def get_prompt_category(filename, summary):
    """欄位 2：Category 提示詞（帶 Summary 上下文）"""
    return f"""基於以下資訊，判斷檔案的分類。

檔案：{filename}
功能描述：{summary}

請從以下選項中選擇「一個」最適合的編號：
0 = 無法判斷（內容不足或模糊）
1 = 交易策略（策略邏輯、回測引擎、訊號生成）
2 = 數據處理（資料載入、清洗、轉換、資料庫操作）
3 = 分析工具（視覺化、報表生成、圖表繪製）
4 = 設定檔案（config, yaml, json 等設定）
5 = 說明文件（README, 教學文件、使用指南）
6 = 研究筆記（研究素材、想法紀錄、筆記）
7 = 測試程式（test_xxx.py, 單元測試）
8 = 過時檔案（舊版本、已棄用、備份檔）
9 = 第三方程式（外部專案、第三方函式庫）

只回答一個數字（0-9）："""

def get_prompt_keep(filename, summary, category):
    """欄位 3：Keep Decision 提示詞（帶完整上下文）"""
    return f"""基於以下資訊，判斷是否應該保留這個檔案。

檔案：{filename}
功能：{summary}
分類：{category}

判斷標準：
- 保留：核心功能、最新版本、有價值的研究、重要文件
- 刪除：重複檔案、舊版本、測試殘留、產出檔案、過時內容
- 0：無法判斷（需要更多資訊）

只回答三個選項之一：「保留」、「刪除」或「0」："""

def get_prompt_reason(filename, summary, category, keep_decision):
    """欄位 4：Reason 提示詞（帶所有前置上下文）"""
    return f"""說明為什麼做出這個決定。

檔案：{filename}
功能：{summary}
分類：{category}
決定：{keep_decision}

要求：
- 用一句話說明理由（不超過 20 字）
- 不要重複說「因為...」，直接說理由
- 如果無法說明，回答：0

範例：
- 「核心回測引擎，持續維護」
- 「舊版已被 v2 取代」
- 「重複檔案，內容相同」

你的理由（不超過20字，無法說明寫0）："""

def get_prompt_confidence(filename, summary, category, keep_decision, reason):
    """欄位 5：Confidence 提示詞（帶完整分析上下文）"""
    return f"""評估你對這次分析的信心程度。

檔案：{filename}
分析結果：
  功能：{summary}
  分類：{category}
  決定：{keep_decision}
  理由：{reason}

信心程度：
A = 高信心（檔案內容清晰，判斷明確可靠）
B = 中信心（內容部分模糊，但判斷合理）
C = 低信心（檔案內容不足，建議人工確認）
0 = 無法評估

只回答 A、B、C 或 0："""

# ===== 執行分析 =====
conn = DB_CONN
cursor = conn.cursor()

# 取得需要分析的檔案（排除重複與已自動分類的核心檔案）
cursor.execute("""
SELECT id, filepath, filename, content_text, auto_category
FROM files
WHERE is_duplicate = 0
  AND id NOT IN (SELECT file_id FROM analysis)
ORDER BY filepath
""")
files_to_analyze = cursor.fetchall()

print(f"\n📊 需分析檔案：{len(files_to_analyze)} 個")
print("⏳ 預估時間：約 {:.0f} 分鐘\n".format(len(files_to_analyze) * 0.5))

total_tokens = 0
start_time = time.time()

for idx, (file_id, filepath, filename, content, auto_category) in enumerate(tqdm(files_to_analyze, desc="分析進度", unit="檔"), 1):

    file_start = time.time()
    file_tokens = 0

    try:
        # ===== 欄位 1: Summary =====
        prompt = get_prompt_summary(filename, filepath, content)
        summary, tokens = ask_model(prompt)
        file_tokens += tokens

        # ===== 欄位 2: Category（帶 Summary 上下文）=====
        prompt = get_prompt_category(filename, summary)
        category_id, tokens = ask_model(prompt)
        file_tokens += tokens

        # 清理並驗證 category_id
        category_id = category_id.strip()
        if category_id not in CATEGORY_MAP:
            category_id = "0"
        category_name = CATEGORY_MAP[category_id]

        # ===== 欄位 3: Keep Decision（帶完整上下文）=====
        prompt = get_prompt_keep(filename, summary, category_name)
        keep_decision, tokens = ask_model(prompt)
        file_tokens += tokens
        keep_decision = keep_decision.strip()

        # ===== 欄位 4: Reason（帶所有前置上下文）=====
        prompt = get_prompt_reason(filename, summary, category_name, keep_decision)
        reason, tokens = ask_model(prompt)
        file_tokens += tokens

        # ===== 欄位 5: Confidence（帶完整分析結果）=====
        prompt = get_prompt_confidence(filename, summary, category_name, keep_decision, reason)
        confidence, tokens = ask_model(prompt)
        file_tokens += tokens
        confidence = confidence.strip().upper()
        if confidence not in ['A', 'B', 'C', '0']:
            confidence = 'B'

        # 計算處理時間
        file_time = time.time() - file_start
        total_tokens += file_tokens

        # 存入資料庫
        cursor.execute("""
        INSERT OR REPLACE INTO analysis
        (file_id, summary, category, category_id, keep_decision, reason,
         confidence, tokens_used, analysis_time_seconds, analyzed_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            file_id, summary, category_name, int(category_id), keep_decision,
            reason, confidence, file_tokens, file_time,
            time.time()
        ))

        # 每 10 個檔案 commit 一次
        if idx % 10 == 0:
            conn.commit()

        # 顯示進度（每 5 個檔案顯示一次詳細資訊）
        if idx % 5 == 0 or idx == 1:
            print(f"\n[{idx}/{len(files_to_analyze)}] {filename}")
            print(f"  📝 {summary}")
            print(f"  🏷️  {category_name} | {keep_decision} | 信心:{confidence}")

    except Exception as e:
        print(f"\n⚠️ 檔案分析失敗：{filename}")
        print(f"   錯誤：{e}")
        # 記錄失敗但繼續
        cursor.execute("""
        INSERT OR REPLACE INTO analysis
        (file_id, summary, category, keep_decision, reason, confidence)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (file_id, "分析失敗", "無法判斷", "0", str(e), "C"))

conn.commit()

# ===== 最終統計 =====
total_time = time.time() - start_time
print("\n" + "=" * 60)
print("📊 分析完成統計：")
print(f"   總檔案：{len(files_to_analyze)}")
print(f"   總耗時：{total_time/60:.1f} 分鐘")
print(f"   平均速度：{total_time/len(files_to_analyze):.1f} 秒/檔")
print(f"   Token 消耗：{total_tokens:,}")
print("=" * 60)

🤖 開始模型分析（逐欄位提問，穩定優先）

🔧 檢查 Ollama 環境...
⬇️ 安裝 Ollama...
✅ Ollama 安裝完成
🚀 啟動 Ollama 伺服器...
✅ Ollama 伺服器已就緒
⬇️ 確保模型 gemma3:4b 已下載...
✅ 模型準備完成

📊 需分析檔案：173 個
⏳ 預估時間：約 86 分鐘



分析進度:   0%|          | 0/173 [00:00<?, ?檔/s]


[1/173] AGENTS.md
  📝 確保程式碼、資料和產出檔案存放在指定目錄下，並使用繁體中文溝通。
  🏷️  設定檔案 | 0 | 信心:B

[5/173] QUANT_STRATEGY_DEVELOPMENT_PLAN.md
  📝 規劃量化台指期貨/選擇權策略開發，包含環境設定、策略回測與組合。
  🏷️  交易策略 | 保留 | 信心:A

[10/173] check_db_schema.py
  📝 此程式碼讀取並顯示taifex.db數據庫的表名、futures表結構及最近五筆資料。
  🏷️  數據處理 | 保留 | 信心:A

[15/173] roro_config.yaml
  📝 配置 API 金鑰、數據來源及資產配置建議，用於 RORO 策略。
  🏷️  設定檔案 | 保留 | 信心:A

[20/173] GITHUB_LARGE_FILES_SOLUTION.md
  📝 解決 GitHub 大檔案上傳問題的三種方案。
  🏷️  說明文件 | 保留 | 信心:A

[25/173] PROJECT_COMMIT_GUIDE.md
  📝 提供量化交易回測框架的提交指南，包含檔案排除策略與提交步驟。
  🏷️  說明文件 | 保留 | 信心:A

[30/173] README.md
  📝 歸檔了 SMA 濾網策略在 SPY 和台指期市場的回測結果與分析。
  🏷️  說明文件 | 保留 | 信心:A

[35/173] 2025-11-27_04-30_roro_stage2_completion_summary.md
  📝 總結了RORO策略階段二的實現，包含風險指標、狀態引擎和信號生成流程。
  🏷️  分析工具 | 保留 | 信心:A

[40/173] 2025-11-24_01-40_root_directory_organization.md
  📝 整理根目錄、修正環境問題，並確保回測程式能正常運作。
  🏷️  設定檔案 | 保留 | 信心:A

[45/173] 2025-11-25_13-08_implement_monthly_buy_and_hold_strategy (1).md
  📝 實作了「月初買入，月底賣出」的月度交易策略，並產生績效報告。
  🏷️  交易策略 | 保留 | 信心

In [ ]:

# @title 📤 階段 4：匯出 CSV 並上傳到 Google Drive

import pandas as pd
import shutil
from datetime import datetime

print("=" * 60)
print("📤 匯出分析結果並上傳到 Google Drive")
print("=" * 60)

conn = DB_CONN
timestamp = TIMESTAMP

# ===== 建立 Drive 目標資料夾 =====
drive_report_root = f"{PROJECT_ROOT}/reports/{timestamp}"
drive_db_dir = f"{drive_report_root}/database"
drive_csv_dir = f"{drive_report_root}/csv"

print(f"\n📁 建立目標資料夾...")
os.makedirs(drive_db_dir, exist_ok=True)
os.makedirs(drive_csv_dir, exist_ok=True)
print(f"✅ {drive_report_root}")

# ===== CSV 1: 檔案清單 =====
print("\n📄 匯出 CSV 1: 檔案清單...")
df_files = pd.read_sql("""
SELECT
    filepath AS 檔案路徑,
    filename AS 檔案名稱,
    file_type AS 副檔名,
    char_count AS 字元數,
    CASE WHEN is_duplicate = 1 THEN '是' ELSE '否' END AS 是否重複,
    auto_category AS 自動分類
FROM files
ORDER BY filepath
""", conn)
csv1_path = f"{drive_csv_dir}/01_files_list.csv"
df_files.to_csv(csv1_path, index=False, encoding='utf-8-sig')
print(f"✅ 已生成：01_files_list.csv ({len(df_files)} 筆)")

# ===== CSV 2: 完整分析結果 =====
print("\n📄 匯出 CSV 2: 完整分析結果...")
df_analysis = pd.read_sql("""
SELECT
    f.filepath AS 檔案路徑,
    f.filename AS 檔案名稱,
    f.file_type AS 副檔名,
    COALESCE(a.summary, '未分析') AS 功能摘要,
    COALESCE(a.category, f.auto_category, '未分類') AS 分類,
    COALESCE(a.keep_decision, '未決定') AS 保留決定,
    COALESCE(a.reason, '') AS 理由,
    COALESCE(a.confidence, '') AS 信心程度,
    f.char_count AS 字元數
FROM files f
LEFT JOIN analysis a ON f.id = a.file_id
WHERE f.is_duplicate = 0
ORDER BY
    CASE
        WHEN a.keep_decision = '刪除' THEN 1
        WHEN a.keep_decision = '保留' THEN 2
        ELSE 3
    END,
    f.filepath
""", conn)
csv2_path = f"{drive_csv_dir}/02_analysis_results.csv"
df_analysis.to_csv(csv2_path, index=False, encoding='utf-8-sig')
print(f"✅ 已生成：02_analysis_results.csv ({len(df_analysis)} 筆)")

# ===== CSV 3: 分類統計摘要 =====
print("\n📄 匯出 CSV 3: 分類統計...")
df_summary = pd.read_sql("""
SELECT
    COALESCE(a.category, f.auto_category, '未分類') AS 分類,
    COUNT(*) AS 檔案數量,
    SUM(CASE WHEN a.keep_decision = '保留' THEN 1 ELSE 0 END) AS 建議保留,
    SUM(CASE WHEN a.keep_decision = '刪除' THEN 1 ELSE 0 END) AS 建議刪除,
    ROUND(AVG(CASE
        WHEN a.confidence = 'A' THEN 3
        WHEN a.confidence = 'B' THEN 2
        WHEN a.confidence = 'C' THEN 1
        ELSE 0
    END), 2) AS 平均信心分數,
    SUM(f.char_count) AS 總字元數
FROM files f
LEFT JOIN analysis a ON f.id = a.file_id
WHERE f.is_duplicate = 0
GROUP BY COALESCE(a.category, f.auto_category, '未分類')
ORDER BY 檔案數量 DESC
""", conn)
csv3_path = f"{drive_csv_dir}/03_summary_report.csv"
df_summary.to_csv(csv3_path, index=False, encoding='utf-8-sig')
print(f"✅ 已生成：03_summary_report.csv ({len(df_summary)} 筆)")

# ===== CSV 4: 建議刪除檔案清單 =====
print("\n📄 匯出 CSV 4: 建議刪除檔案...")
df_delete = pd.read_sql("""
SELECT
    f.filepath AS 檔案路徑,
    f.filename AS 檔案名稱,
    a.category AS 分類,
    a.reason AS 刪除理由,
    a.confidence AS 信心程度
FROM files f
JOIN analysis a ON f.id = a.file_id
WHERE a.keep_decision = '刪除'
ORDER BY a.category, f.filepath
""", conn)
csv4_path = f"{drive_csv_dir}/04_suggested_deletions.csv"
df_delete.to_csv(csv4_path, index=False, encoding='utf-8-sig')
print(f"✅ 已生成：04_suggested_deletions.csv ({len(df_delete)} 筆)")

# ===== 複製資料庫到 Drive =====
print("\n🗄️ 複製 SQLite 資料庫...")
db_target_path = f"{drive_db_dir}/project_analysis.db"
shutil.copy2(DB_PATH, db_target_path)
print(f"✅ 資料庫已複製到：{db_target_path}")

# ===== 最終統計顯示 =====
print("\n" + "=" * 60)
print("📊 專案分析最終報告")
print("=" * 60)
print(f"\n📍 報告位置：{drive_report_root}")
print(f"\n📁 資料夾結構：")
print(f"   ├── database/")
print(f"   │   └── project_analysis.db")
print(f"   └── csv/")
print(f"       ├── 01_files_list.csv")
print(f"       ├── 02_analysis_results.csv")
print(f"       ├── 03_summary_report.csv")
print(f"       └── 04_suggested_deletions.csv")

print(f"\n📈 統計摘要：")
print(df_summary.to_string(index=False))

print(f"\n🎯 重點提醒：")
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM files WHERE is_duplicate = 0")
total_files = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM analysis WHERE keep_decision = '刪除'")
delete_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM analysis WHERE keep_decision = '保留'")
keep_count = cursor.fetchone()[0]

print(f"   - 總檔案數：{total_files}")
print(f"   - 建議保留：{keep_count} ({keep_count/total_files*100:.1f}%)")
print(f"   - 建議刪除：{delete_count} ({delete_count/total_files*100:.1f}%)")

print("\n" + "=" * 60)
print("🎉 全部流程完成！")
print("=" * 60)

📤 匯出分析結果並上傳到 Google Drive

📁 建立目標資料夾...
✅ /content/drive/MyDrive/sp_lab_v10 - G/reports/2025-11-27T22:06:59+08:00

📄 匯出 CSV 1: 檔案清單...
✅ 已生成：01_files_list.csv (177 筆)

📄 匯出 CSV 2: 完整分析結果...
✅ 已生成：02_analysis_results.csv (173 筆)

📄 匯出 CSV 3: 分類統計...
✅ 已生成：03_summary_report.csv (9 筆)

📄 匯出 CSV 4: 建議刪除檔案...
✅ 已生成：04_suggested_deletions.csv (3 筆)

🗄️ 複製 SQLite 資料庫...
✅ 資料庫已複製到：/content/drive/MyDrive/sp_lab_v10 - G/reports/2025-11-27T22:06:59+08:00/database/project_analysis.db

📊 專案分析最終報告

📍 報告位置：/content/drive/MyDrive/sp_lab_v10 - G/reports/2025-11-27T22:06:59+08:00

📁 資料夾結構：
   ├── database/
   │   └── project_analysis.db
   └── csv/
       ├── 01_files_list.csv
       ├── 02_analysis_results.csv
       ├── 03_summary_report.csv
       └── 04_suggested_deletions.csv

📈 統計摘要：
   分類  檔案數量  建議保留  建議刪除  平均信心分數   總字元數
 數據處理    53    27     0    2.21 321328
 交易策略    47    22     2    2.06 390768
 說明文件    28    17     0    1.96  79417
 分析工具    13     8     1    2.46  60749
 設定檔案    12     7     

In [6]:

# @title 🤖 階段 3：逐欄位模型分析（滾動視窗）

import time
import requests
import subprocess
from datetime import datetime
import pytz
from collections import deque
from IPython.display import clear_output

print("=" * 60)
print("🤖 模型分析系統啟動")
print("=" * 60)

# ===== 使用者參數設定 =====
# @markdown ### 📋 模型選擇
MODEL_SELECTION = "qwen3:4b" # @param ["gemma3:270m", "gemma3:1b", "gemma3:4b", "qwen3:0.5b", "qwen3:2b", "qwen3:4b", "qwen3:8b", "qwen2.5-coder:1.5b", "qwen2.5-coder:3b", "qwen2.5-coder:7b"]

# @markdown ### ⚙️ 模型參數（進階）
# @markdown 溫度（0.0-1.0）：數值越低，輸出越穩定
TEMPERATURE = "0.1" # @param {type:"string"}

# @markdown 上下文長度（tokens）：模型可處理的最大文字長度
NUM_CTX = "8192" # @param {type:"string"}

# @markdown 單次超時時間（秒）：單個檔案分析的最長等待時間
TIMEOUT_SECONDS = "120" # @param {type:"string"}

# @markdown ### 📺 顯示設定
# @markdown 視窗保留行數：畫面上顯示最新幾個檔案
DISPLAY_WINDOW_SIZE = "30" # @param {type:"string"}

# @markdown 是否記錄詳細 log 檔
ENABLE_DETAILED_LOG = True # @param {type:"boolean"}

# ===== 內部參數（可在程式碼中調整）=====
INTERNAL_CONFIG = {
    'show_avg_speed': False,          # 是否顯示平均速度
    'show_cumulative_tokens': True,   # 是否顯示累計 Token
    'show_estimated_time': True,      # 是否顯示預估剩餘時間
    'highlight_deletions': True,      # 是否用 💡 標示建議刪除
    'screen_timestamp_format': 'HMS', # 畫面時間格式：'HMS'=HH:MM:SS, 'HMSM'=HH:MM:SS.mmm
    'log_file_path': '/content/analysis_detailed.log',  # Log 檔路徑
}

# ===== 參數轉換與驗證 =====
try:
    TEMPERATURE = float(TEMPERATURE)
    NUM_CTX = int(NUM_CTX)
    TIMEOUT_SECONDS = int(TIMEOUT_SECONDS)
    DISPLAY_WINDOW_SIZE = int(DISPLAY_WINDOW_SIZE)
except ValueError as e:
    print(f"❌ 參數格式錯誤：{e}")
    print("請確認所有參數都是有效數值")
    raise

print(f"\n✅ 參數設定：")
print(f"   模型：{MODEL_SELECTION}")
print(f"   溫度：{TEMPERATURE}")
print(f"   上下文：{NUM_CTX}")
print(f"   超時：{TIMEOUT_SECONDS}s")
print(f"   視窗大小：{DISPLAY_WINDOW_SIZE} 行")

# ===== 時區設定 =====
taipei_tz = pytz.timezone('Asia/Taipei')

def get_timestamp(format_type='iso_full'):
    """取得台北時區時間戳"""
    now = datetime.now(taipei_tz)
    if format_type == 'iso_full':
        # 完整 ISO 8601（毫秒級）
        return now.isoformat(timespec='milliseconds')
    elif format_type == 'hms':
        # 時:分:秒
        return now.strftime('%H:%M:%S')
    elif format_type == 'hmsm':
        # 時:分:秒.毫秒
        return now.strftime('%H:%M:%S.%f')[:-3]
    else:
        return now.isoformat()

# ===== Ollama 環境檢查與設定 =====
def ensure_ollama():
    """確保 Ollama 已安裝並啟動"""
    print("\n🔧 檢查 Ollama 環境...")

    # 檢查是否已安裝
    try:
        result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=5)
        print(f"✅ Ollama 已安裝：{result.stdout.strip()}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print("⬇️ 安裝 Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        print("✅ Ollama 安裝完成")

    # 啟動伺服器
    print("🚀 啟動 Ollama 伺服器...")
    subprocess.run("pkill ollama", shell=True, stderr=subprocess.DEVNULL)
    time.sleep(2)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

    # 確認伺服器就緒
    for attempt in range(10):
        try:
            requests.get("http://127.0.0.1:11434", timeout=2)
            print("✅ Ollama 伺服器已就緒")
            break
        except:
            if attempt == 9:
                raise RuntimeError("Ollama 伺服器啟動失敗")
            time.sleep(2)

    # 下載模型
    print(f"⬇️ 確保模型 {MODEL_SELECTION} 已下載...")
    result = subprocess.run(["ollama", "pull", MODEL_SELECTION], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ 模型準備完成")
    else:
        print(f"⚠️ 模型下載警告：{result.stderr}")

ensure_ollama()

# ===== 模型呼叫函式 =====
def ask_model(prompt, max_retries=3):
    """呼叫 Ollama 模型並返回清理後的回答"""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                "http://127.0.0.1:11434/api/generate",
                json={
                    "model": MODEL_SELECTION,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": TEMPERATURE,
                        "num_ctx": NUM_CTX,
                    }
                },
                timeout=TIMEOUT_SECONDS
            )
            response.raise_for_status()
            data = response.json()
            answer = data.get("response", "").strip()
            tokens = data.get("eval_count", 0)
            return answer, tokens
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"\n⚠️ 模型呼叫失敗（已重試{max_retries}次）：{e}")
                return "0", 0
            time.sleep(2)
    return "0", 0

# ===== 分類對照表 =====
CATEGORY_MAP = {
    "0": "無法判斷",
    "1": "交易策略",
    "2": "數據處理",
    "3": "分析工具",
    "4": "設定檔案",
    "5": "說明文件",
    "6": "研究筆記",
    "7": "測試程式",
    "8": "過時檔案",
    "9": "第三方程式",
}

# ===== 提示詞設計 =====
def get_prompt_summary(filename, filepath, content):
    return f"""你是專業的程式碼分析助手。請用一句繁體中文描述這個檔案的主要功能。

要求：
- 不超過 30 個字
- 不要說「這個檔案是...」開頭，直接說功能
- 如果無法判斷，回答：0

檔案名稱：{filename}
檔案路徑：{filepath}
檔案內容（前 2000 字）：
{content[:2000]}

你的一句話描述（不超過30字，無法判斷寫0）："""

def get_prompt_category(filename, summary):
    return f"""基於以下資訊，判斷檔案的分類。

檔案：{filename}
功能描述：{summary}

請從以下選項中選擇「一個」最適合的編號：
0 = 無法判斷（內容不足或模糊）
1 = 交易策略（策略邏輯、回測引擎、訊號生成）
2 = 數據處理（資料載入、清洗、轉換、資料庫操作）
3 = 分析工具（視覺化、報表生成、圖表繪製）
4 = 設定檔案（config, yaml, json 等設定）
5 = 說明文件（README, 教學文件、使用指南）
6 = 研究筆記（研究素材、想法紀錄、筆記）
7 = 測試程式（test_xxx.py, 單元測試）
8 = 過時檔案（舊版本、已棄用、備份檔）
9 = 第三方程式（外部專案、第三方函式庫）

只回答一個數字（0-9）："""

def get_prompt_keep(filename, summary, category):
    return f"""基於以下資訊，判斷是否應該保留這個檔案。

檔案：{filename}
功能：{summary}
分類：{category}

判斷標準：
- 保留：核心功能、最新版本、有價值的研究、重要文件
- 刪除：重複檔案、舊版本、測試殘留、產出檔案、過時內容
- 0：無法判斷（需要更多資訊）

只回答三個選項之一：「保留」、「刪除」或「0」："""

def get_prompt_reason(filename, summary, category, keep_decision):
    return f"""說明為什麼做出這個決定。

檔案：{filename}
功能：{summary}
分類：{category}
決定：{keep_decision}

要求：
- 用一句話說明理由（不超過 20 字）
- 不要重複說「因為...」，直接說理由
- 如果無法說明，回答：0

範例：
- 「核心回測引擎，持續維護」
- 「舊版已被 v2 取代」
- 「重複檔案，內容相同」

你的理由（不超過20字，無法說明寫0）："""

def get_prompt_confidence(filename, summary, category, keep_decision, reason):
    return f"""評估你對這次分析的信心程度。

檔案：{filename}
分析結果：
  功能：{summary}
  分類：{category}
  決定：{keep_decision}
  理由：{reason}

信心程度：
A = 高信心（檔案內容清晰，判斷明確可靠）
B = 中信心（內容部分模糊，但判斷合理）
C = 低信心（檔案內容不足，建議人工確認）
0 = 無法評估

只回答 A、B、C 或 0："""

# ===== 顯示格式化函式 =====
def format_file_display(idx, total, filename, summary, category, keep, reason, confidence, file_time, file_tokens, total_tokens):
    """格式化單個檔案的顯示內容"""
    timestamp = get_timestamp(INTERNAL_CONFIG['screen_timestamp_format'].lower())

    lines = []
    lines.append(f"[{timestamp}] [{idx}/{total}] {filename}")
    lines.append(f"  📝 {summary}")

    # 決定顏色標示
    keep_emoji = ""
    if INTERNAL_CONFIG['highlight_deletions'] and keep == "刪除":
        keep_emoji = " 💡"

    lines.append(f"  🏷️  {category} | {keep}{keep_emoji} | 信心:{confidence}")

    # 理由（只在建議刪除時顯示）
    if keep == "刪除" and reason != "0":
        lines.append(f"  💡 理由：{reason}")

    # 統計資訊
    stats_parts = [f"⏱️ {file_time:.1f}s"]
    if INTERNAL_CONFIG['show_cumulative_tokens']:
        stats_parts.append(f"🔢 {file_tokens}t (總計:{total_tokens}t)")
    else:
        stats_parts.append(f"🔢 {file_tokens}t")

    lines.append(f"  {' | '.join(stats_parts)}")

    return "\n".join(lines)

def format_log_entry(timestamp_iso, idx, total, filename, summary, category, category_id, keep, reason, confidence, file_time, file_tokens, total_tokens):
    """格式化 Log 檔條目"""
    return f"""[{timestamp_iso}] [{idx}/{total}] {filename}
  Summary: {summary}
  Category: {category} ({category_id})
  Keep: {keep}
  Reason: {reason}
  Confidence: {confidence}
  Time: {file_time:.3f}s
  Tokens: {file_tokens}
  Cumulative_Tokens: {total_tokens}
  Status: OK
"""

# ===== 初始化 Log 檔 =====
if ENABLE_DETAILED_LOG:
    log_file = open(INTERNAL_CONFIG['log_file_path'], 'w', encoding='utf-8')
    log_file.write("=" * 60 + "\n")
    log_file.write("專案分析詳細記錄\n")
    log_file.write(f"開始時間: {get_timestamp('iso_full')}\n")
    log_file.write(f"模型: {MODEL_SELECTION} | 溫度: {TEMPERATURE} | 上下文: {NUM_CTX}\n")
    log_file.write("=" * 60 + "\n\n")
    log_file.flush()

# ===== 執行分析 =====
conn = DB_CONN
cursor = conn.cursor()

# 取得需要分析的檔案
cursor.execute("""
SELECT id, filepath, filename, content_text, auto_category
FROM files
WHERE is_duplicate = 0
  AND id NOT IN (SELECT file_id FROM analysis)
ORDER BY filepath
""")
files_to_analyze = cursor.fetchall()

total_files = len(files_to_analyze)
print(f"\n📊 需分析檔案：{total_files} 個")
print(f"📺 視窗大小：{DISPLAY_WINDOW_SIZE} 行")
print(f"💾 Log 檔：{INTERNAL_CONFIG['log_file_path'] if ENABLE_DETAILED_LOG else '未啟用'}\n")

# 初始化變數
display_buffer = deque(maxlen=DISPLAY_WINDOW_SIZE)
total_tokens = 0
start_time = time.time()

# 主處理迴圈
for idx, (file_id, filepath, filename, content, auto_category) in enumerate(files_to_analyze, 1):

    file_start = time.time()
    file_tokens = 0

    try:
        # ===== 欄位 1: Summary =====
        prompt = get_prompt_summary(filename, filepath, content)
        summary, tokens = ask_model(prompt)
        file_tokens += tokens

        # ===== 欄位 2: Category =====
        prompt = get_prompt_category(filename, summary)
        category_id, tokens = ask_model(prompt)
        file_tokens += tokens
        category_id = category_id.strip()
        if category_id not in CATEGORY_MAP:
            category_id = "0"
        category_name = CATEGORY_MAP[category_id]

        # ===== 欄位 3: Keep Decision =====
        prompt = get_prompt_keep(filename, summary, category_name)
        keep_decision, tokens = ask_model(prompt)
        file_tokens += tokens
        keep_decision = keep_decision.strip()

        # ===== 欄位 4: Reason =====
        prompt = get_prompt_reason(filename, summary, category_name, keep_decision)
        reason, tokens = ask_model(prompt)
        file_tokens += tokens

        # ===== 欄位 5: Confidence =====
        prompt = get_prompt_confidence(filename, summary, category_name, keep_decision, reason)
        confidence, tokens = ask_model(prompt)
        file_tokens += tokens
        confidence = confidence.strip().upper()
        if confidence not in ['A', 'B', 'C', '0']:
            confidence = 'B'

        # 計算處理時間
        file_time = time.time() - file_start
        total_tokens += file_tokens

        # 存入資料庫
        cursor.execute("""
        INSERT OR REPLACE INTO analysis
        (file_id, summary, category, category_id, keep_decision, reason,
         confidence, tokens_used, analysis_time_seconds, analyzed_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            file_id, summary, category_name, int(category_id), keep_decision,
            reason, confidence, file_tokens, file_time, time.time()
        ))

        # 每 10 個檔案 commit 一次
        if idx % 10 == 0:
            conn.commit()

        # 寫入 Log 檔
        if ENABLE_DETAILED_LOG:
            log_entry = format_log_entry(
                get_timestamp('iso_full'), idx, total_files, filename,
                summary, category_name, category_id, keep_decision, reason,
                confidence, file_time, file_tokens, total_tokens
            )
            log_file.write(log_entry + "\n")
            log_file.flush()

        # 格式化顯示內容並加入緩衝區
        display_text = format_file_display(
            idx, total_files, filename, summary, category_name,
            keep_decision, reason, confidence, file_time, file_tokens, total_tokens
        )
        display_buffer.append(display_text)

    except Exception as e:
        print(f"\n⚠️ 檔案分析失敗：{filename}")
        print(f"   錯誤：{e}")
        cursor.execute("""
        INSERT OR REPLACE INTO analysis
        (file_id, summary, category, keep_decision, reason, confidence)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (file_id, "分析失敗", "無法判斷", "0", str(e), "C"))
        display_buffer.append(f"[ERROR] {filename}: {e}")

    # ===== 重繪畫面（滾動窗口）=====
    clear_output(wait=True)

    # 標題區
    print("=" * 60)
    print(f"🤖 量化專案分析中")
    print(f"模型: {MODEL_SELECTION} | 溫度: {TEMPERATURE} | 已完成: {idx}/{total_files} ({idx/total_files*100:.1f}%)")
    print("=" * 60)
    print()

    # 顯示緩衝區內容（最新 N 行）
    for log_line in display_buffer:
        print(log_line)
        print()  # 檔案間空行

    # 底部狀態列
    print("=" * 60)

    # 計算統計資訊
    elapsed = time.time() - start_time
    if idx > 0:
        avg_time = elapsed / idx
        remaining = (total_files - idx) * avg_time
        mins, secs = divmod(int(remaining), 60)

        status_parts = []
        if INTERNAL_CONFIG['show_estimated_time']:
            status_parts.append(f"⏱️ 預估剩餘: {mins}分{secs}秒")
        if INTERNAL_CONFIG['show_cumulative_tokens']:
            token_rate = total_tokens / elapsed if elapsed > 0 else 0
            status_parts.append(f"📊 Token速度: {token_rate:.0f}t/s")
        if ENABLE_DETAILED_LOG:
            status_parts.append(f"💾 Log: {INTERNAL_CONFIG['log_file_path']}")

        print(" | ".join(status_parts))

    print("=" * 60)

# ===== 最終提交與清理 =====
conn.commit()

if ENABLE_DETAILED_LOG:
    log_file.write("\n" + "=" * 60 + "\n")
    log_file.write(f"分析完成時間: {get_timestamp('iso_full')}\n")
    total_time = time.time() - start_time
    mins, secs = divmod(int(total_time), 60)
    log_file.write(f"總耗時: {mins}分{secs}秒\n")
    log_file.write("=" * 60 + "\n")
    log_file.close()
    print(f"\n✅ 詳細記錄已儲存：{INTERNAL_CONFIG['log_file_path']}")

# ===== 最終統計 =====
print("\n" + "=" * 60)
print("📊 分析完成統計")
print("=" * 60)
print(f"總檔案：{total_files}")
total_time = time.time() - start_time
mins, secs = divmod(int(total_time), 60)
print(f"總耗時：{mins}分{secs}秒")
print(f"Token 消耗：{total_tokens:,}")
print("=" * 60)

🤖 模型分析系統啟動

✅ 參數設定：
   模型：qwen3:4b
   溫度：0.1
   上下文：8192
   超時：120s
   視窗大小：30 行

🔧 檢查 Ollama 環境...
✅ Ollama 已安裝：Warning: could not connect to a running Ollama instance
🚀 啟動 Ollama 伺服器...
✅ Ollama 伺服器已就緒
⬇️ 確保模型 qwen3:4b 已下載...
✅ 模型準備完成

📊 需分析檔案：167 個
📺 視窗大小：30 行
💾 Log 檔：/content/analysis_detailed.log



KeyboardInterrupt: 